# Cross-Dataset Gene Outlier Consistency

- **Human datasets**: A3_H1 (PBMCs rep1), A3_H2 (PBMCs rep2), A5 (Frozen PBMCs), A6 (Cancer cells), A7 (Frozen PBMCs)
- **PBMC datasets**: A3_H1, A3_H2, A5, A7

In [1]:
%matplotlib inline

In [2]:
# Import packages
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from upsetty import Upset
from XvP_utils import plotting
import gseapy as gp
import matplotlib.patches as mpatches
from scipy.stats import gmean
import re

project_dir = Path("/home/mcaskey/10XvParse/")

tenx_outlier_out_dir = Path("10x_outliers")
parse_outlier_out_dir = Path("parse_outliers")
tenx_outlier_out_dir.mkdir(exist_ok=True)
parse_outlier_out_dir.mkdir(exist_ok=True)


datasets = {
    "A3_H1": "Notebooks/Analysis_3/gene_data/gene_comparisons_H1.csv",
    "A3_H2": "Notebooks/Analysis_3/gene_data/gene_comparisons_H2.csv",
    "A5": "Notebooks/Analysis_5/gene_data/gene_comparisons.csv",
    "A6": "Notebooks/Analysis_6/gene_data/gene_comparisons.csv",
    "A7": "Notebooks/Analysis_7/gene_data/gene_comparisons.csv"
}

# Load data
data = {}
for name, path in datasets.items():
    data[name] = pd.read_csv(project_dir / path)
    data[name].drop(columns=["Unnamed: 0"], inplace=True)

# Remove genes with low percent counts in all datasets
for name, df in data.items():
    pct_threshold = 0.001
    drop_indices = df[(df["parse_percent_counts"] < pct_threshold) & (df["10x_percent_counts"] < pct_threshold)].index
    df.drop(index=drop_indices, inplace=True)

# Find outlier genes
for name, df in data.items():
    pct_threshold = 0.5
    CLR = np.log2((df["10x_percent_counts"] + 1e-6) / (df["parse_percent_counts"] + 1e-6)) - np.log2(gmean(df['10x_percent_counts'] + 1e-6) / gmean(df['parse_percent_counts'] + 1e-6))
    df["10x_outliers"] = CLR >= np.log2(1 + pct_threshold)  # Outliers are genes with >10% higher percent counts in 10x
    df["parse_outliers"] = CLR <= np.log2(1/(1+pct_threshold))  # Outliers are genes with >10% higher percent counts in parse

# Suffix dataset specific columns
for name, df in data.items():
    for col_name in df.columns:
        if col_name.endswith("_distance") or col_name.endswith("_percent_counts") or col_name.endswith("_n_cells") or col_name.endswith("_outliers"):
            new_col_name = f"{name}_{col_name}"
            df.rename(columns={col_name: new_col_name}, inplace=True)

merged_df = pd.DataFrame()
for name, df in data.items():
    
    if merged_df.empty:
        merged_df = df
    else:
        merged_df = merged_df.merge(df, on=["gene_name", "gene_id", "gene_length", "gc_content",
                                     "is_lnc", "is_pc", "is_mito", "is_ribo"], how="outer")

for col_name in merged_df.columns:
    if col_name.endswith("_distance") or col_name.endswith("_percent_counts") or col_name.endswith("_n_cells"):
        merged_df[col_name] = merged_df[col_name].fillna(0)
    if col_name.endswith("_outliers"):
        merged_df[col_name] = merged_df[col_name].fillna(False)
merged_df.set_index("gene_name", inplace=True)

/tmp/ipykernel_222554/3673265207.py:68: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  merged_df[col_name] = merged_df[col_name].fillna(False)


In [3]:
tenx_outlier_data = merged_df[[col for col in merged_df.columns if col.endswith("_10x_outliers")]]
tenx_outlier_data.columns = [col.replace("_10x_outliers", "") for col in tenx_outlier_data.columns]
tenx_outlier_data = tenx_outlier_data.astype(bool)
tenx_outlier_data = tenx_outlier_data[tenx_outlier_data.any(axis=1)]
upset = Upset.generate_plot(tenx_outlier_data)
upset.show()

In [4]:
# Extract gene subsets that were interesting in the upset plot

tenx_outlier_overlap = tenx_outlier_data.index[tenx_outlier_data['A3_H1']& 
                                                tenx_outlier_data['A3_H2']&
                                                tenx_outlier_data['A5']&
                                                tenx_outlier_data['A6']&
                                                tenx_outlier_data['A7']]
tenx_outlier_overlap = tenx_outlier_overlap.tolist()

name = '10X Outlier Genes Shared by All Analyses'
gene_sets = [tenx_outlier_overlap]
cols = ['is_pc', 'is_lnc', 'is_mito', 'is_ribo']
col_names = ['protein-coding count', 'lncRNA count', 'mtRNA count', 'rRNA count']
color = ['yellow', 'blue', 'red', 'green']

background_lengths = merged_df['gene_length'].tolist()
background_gcs = merged_df['gc_content'].tolist()
lengths = merged_df[merged_df.index.isin(tenx_outlier_overlap)]['gene_length'].tolist()
gcs = merged_df[merged_df.index.isin(tenx_outlier_overlap)]['gc_content'].tolist()

In [5]:
fig, ax = plt.subplots(1, 2, figsize=(12, 6), sharey=True)
ax[0].violinplot(np.log10(lengths), showextrema=False, showmedians=True)
ax[1].violinplot(np.log10(background_lengths), showextrema=False, showmedians=True)
ax[0].set_title(name, fontsize=10)
ax[1].set_title("Background", fontsize=10)
ax[0].set_ylabel("Gene Length (log10)", fontsize=10)
ax[0].set_xticks([])
ax[1].set_xticks([])
plt.savefig("10x_outliers/gene_length_violin.png")
plt.tight_layout()
plt.show()

In [6]:
fig, ax = plt.subplots(1, 2, figsize=(12, 6), sharey=True)
ax[0].violinplot(gcs, showextrema=False, showmedians=True)
ax[1].violinplot(background_gcs, showextrema=False, showmedians=True)
ax[0].set_title(name, fontsize=10)
ax[1].set_title("Background", fontsize=10)
ax[0].set_ylabel("Percent GC Content)", fontsize=10)
ax[0].set_xticks([])
ax[1].set_xticks([])
plt.savefig("10x_outliers/gc_content_violin.png")
plt.tight_layout()
plt.show()

In [7]:
fig, ax = plt.subplots(1,2, figsize=(12, 6))
sums = []
pcts = []
for col in cols:
    sum = merged_df[col][merged_df.index.isin(tenx_outlier_overlap)].sum()
    sums.append(sum)
    pcts.append(sum/merged_df[col].sum())

ax[0].bar(col_names, sums, color=color)
ax[1].bar(col_names, pcts, color=color)

ax[0].set_ylabel('Count of Genes in Type')
ax[1].set_ylabel('Percent of Genes in Type')

fig.suptitle(name)

plt.tight_layout()
plt.savefig("10x_outliers/gene_type_bar.png")
plt.show()

In [8]:
# Get genes consistently enriched in 10X across all 5 datasets
all5_10x = merged_df[
    merged_df[[c for c in merged_df.columns if c.endswith('_10x_outliers')]].all(axis=1)
].index.tolist()

# Background = all genes in your merged table
background = merged_df.index.tolist()

enr = gp.enrichr(
    gene_list=all5_10x,
    gene_sets=['GO_Biological_Process_2023', 'GO_Molecular_Function_2023',
               'KEGG_2021_Human', 'MSigDB_Hallmark_2020'],
    background=background,   # important — use your detected genes, not all human genes
    outdir=None,
    verbose=False,
)

tenx_ern_results = enr.results[enr.results['Adjusted P-value'] < 0.05].sort_values('Adjusted P-value')
tenx_ern_results.to_csv("10x_outliers/enrichment_results.csv")

In [9]:
def extract_prefix(gene_name):
    # Leading letters up to first digit or hyphen: RPS27→RPS, MT-CO1→MT, ATP5F1A→ATP
    m = re.match(r'^([A-Z]+)', gene_name)
    return m.group(1) if m else gene_name

prefixes = pd.Series(all5_10x).apply(extract_prefix)
prefix_counts = prefixes.value_counts()

with open("10x_outliers/enriched_prefixes.txt", "w") as f:
    for prefix in sorted(prefix_counts.index.tolist()):
        f.write(f"{prefix}\n")

prefix_counts_filtered = prefix_counts[prefix_counts > 5].sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, max(4, len(prefix_counts_filtered) * 0.35)))
ax.barh(prefix_counts_filtered.index[::-1], prefix_counts_filtered.values[::-1])
ax.set_xlabel("Number of genes in all-5-dataset 10X-enriched set")
ax.set_title(f"Gene name prefixes (n={len(all5_10x)} total, prefixes with >5 genes)")
plt.tight_layout()
plt.savefig("10x_outliers/gene_prefix_bar.png", dpi=150)
plt.show()

In [10]:
parse_outlier_data = merged_df[[col for col in merged_df.columns if col.endswith("_parse_outliers")]]
parse_outlier_data.columns = [col.replace("_parse_outliers", "") for col in parse_outlier_data.columns]
parse_outlier_data = parse_outlier_data.astype(bool)
parse_outlier_data = parse_outlier_data[parse_outlier_data.any(axis=1)]
upset = Upset.generate_plot(parse_outlier_data)
upset.show()

In [11]:
# Extract gene subsets that were interesting in the upset plot

parse_outlier_overlap = parse_outlier_data.index[parse_outlier_data['A3_H1']& 
                                                    parse_outlier_data['A3_H2']&
                                                    parse_outlier_data['A5']&
                                                    parse_outlier_data['A6']&
                                                    parse_outlier_data['A7']]
parse_outlier_overlap = parse_outlier_overlap.tolist()

name = 'Parse Outlier Genes Shared by All Analyses'
gene_sets = [parse_outlier_overlap]
cols = ['is_pc', 'is_lnc', 'is_mito', 'is_ribo']
col_names = ['protein-coding count', 'lncRNA count', 'mtRNA count', 'rRNA count']
color = ['yellow', 'blue', 'red', 'green']

background_lengths = merged_df['gene_length'].tolist()
background_gcs = merged_df['gc_content'].tolist()
lengths = merged_df[merged_df.index.isin(parse_outlier_overlap)]['gene_length'].tolist()
gcs = merged_df[merged_df.index.isin(parse_outlier_overlap)]['gc_content'].tolist()

In [12]:
fig, ax = plt.subplots(1, 2, figsize=(12, 6), sharey=True)
ax[0].violinplot(np.log10(lengths), showextrema=False, showmedians=True)
ax[1].violinplot(np.log10(background_lengths), showextrema=False, showmedians=True)
ax[0].set_title(name, fontsize=10)
ax[1].set_title("Background", fontsize=10)
ax[0].set_ylabel("Gene Length (log10)", fontsize=10)
ax[0].set_xticks([])
ax[1].set_xticks([])
plt.savefig("parse_outliers/gene_length_violin.png")
plt.tight_layout()
plt.show()

In [13]:
fig, ax = plt.subplots(1, 2, figsize=(12, 6), sharey=True)
ax[0].violinplot(gcs, showextrema=False, showmedians=True)
ax[1].violinplot(background_gcs, showextrema=False, showmedians=True)
ax[0].set_title(name, fontsize=10)
ax[1].set_title("Background", fontsize=10)
ax[0].set_ylabel("Percent GC Content)", fontsize=10)
ax[0].set_xticks([])
ax[1].set_xticks([])
plt.savefig("parse_outliers/gc_content_violin.png")
plt.tight_layout()
plt.show()

In [14]:
fig, ax = plt.subplots(1,2, figsize=(12, 6))
sums = []
pcts = []
for col in cols:
    sum = merged_df[col][merged_df.index.isin(parse_outlier_overlap)].sum()
    sums.append(sum)
    pcts.append(sum/merged_df[col].sum())

ax[0].bar(col_names, sums, color=color)
ax[1].bar(col_names, pcts, color=color)

ax[0].set_ylabel('Count of Genes in Type')
ax[1].set_ylabel('Percent of Genes in Type')

fig.suptitle(name)

plt.tight_layout()
plt.savefig("parse_outliers/gene_type_bar.png")
plt.show()

In [15]:
# Get genes consistently enriched in 10X across all 5 datasets
all5_parse = merged_df[
    merged_df[[c for c in merged_df.columns if c.endswith('_parse_outliers')]].all(axis=1)
].index.tolist()

# Background = all genes in your merged table
background = merged_df.index.tolist()

enr = gp.enrichr(
    gene_list=all5_parse,
    gene_sets=['GO_Biological_Process_2023', 'GO_Molecular_Function_2023',
               'KEGG_2021_Human', 'MSigDB_Hallmark_2020'],
    background=background,   # important — use your detected genes, not all human genes
    outdir=None,
    verbose=False,
)

parse_ern_results = enr.results[enr.results['Adjusted P-value'] < 0.05].sort_values('Adjusted P-value')
parse_ern_results.to_csv("parse_outliers/enrichment_results.csv")

In [16]:
DB_COLORS = {
    'GO_Biological_Process_2023': '#4e9af1',
    'GO_Molecular_Function_2023': '#7bc67e',
    'KEGG_2021_Human':            '#f4a261',
    'MSigDB_Hallmark_2020':       '#e76f51',
}

def shorten(term, maxlen=48):
    term = term.split(' (GO:')[0].strip()
    return term[:maxlen] + '…' if len(term) > maxlen else term

def top_terms(df, n_per_db=4):
    sig = df[df['Adjusted P-value'] < 0.05].copy()
    sig['nlp'] = -np.log10(sig['Adjusted P-value'])
    sig['label'] = sig['Term'].apply(shorten)
    return (sig.groupby('Gene_set', group_keys=False)
               .apply(lambda g: g.nlargest(n_per_db, 'nlp'))
               .reset_index(drop=True))

tenx_top  = top_terms(tenx_ern_results).sort_values('nlp')
parse_top = top_terms(parse_ern_results).sort_values('nlp')

n_t, n_p = len(tenx_top), len(parse_top)
gap = 1  # blank row between the two halves

# y positions: parse at bottom, gap, 10X at top
y_parse = np.arange(n_p)
y_tenx  = np.arange(n_p + gap, n_p + gap + n_t)

fig, ax = plt.subplots(figsize=(13, (n_t + n_p) * 0.38 + 2))

ax.barh(y_tenx,  tenx_top['nlp'],
        color=[DB_COLORS[d] for d in tenx_top['Gene_set']],  alpha=0.85)
ax.barh(y_parse, -parse_top['nlp'],
        color=[DB_COLORS[d] for d in parse_top['Gene_set']], alpha=0.85)

# tick labels
all_y      = np.concatenate([y_parse, y_tenx])
all_labels = np.concatenate([parse_top['label'].values, tenx_top['label'].values])
ax.set_yticks(all_y)
ax.set_yticklabels(all_labels, fontsize=8.5)

ax.axvline(0, color='black', linewidth=0.8)
ax.axhline(n_p + gap/2 - 0.5, color='#888888', linewidth=0.6, linestyle='--')

# section labels
xlim = max(tenx_top['nlp'].max(), parse_top['nlp'].max())
ax.text( xlim * 0.02, y_tenx.mean(),  '← 10X enriched',  va='center', fontweight='bold', fontsize=9)
ax.text(-xlim * 0.02, y_parse.mean(), 'Parse enriched →', va='center', ha='right', fontweight='bold', fontsize=9)

ax.set_xlabel('−log₁₀(adjusted p-value)')
ax.set_title('Functional enrichment: genes consistently enriched in 10X vs Parse\n(all human datasets, log₂FC ≥ log₂1.1 threshold)', fontsize=10)

db_labels = {'GO_Biological_Process_2023': 'GO Biological Process',
             'GO_Molecular_Function_2023':  'GO Molecular Function',
             'KEGG_2021_Human':             'KEGG',
             'MSigDB_Hallmark_2020':        'MSigDB Hallmark'}
ax.legend(handles=[mpatches.Patch(facecolor=c, label=db_labels[d]) for d, c in DB_COLORS.items()],
          fontsize=8, title='Database', loc='lower right')

plt.tight_layout()
plt.savefig("10x_vs_parse_enrichment_bar.png", dpi=300)
plt.show()


/tmp/ipykernel_222554/1651739286.py:17: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.

/tmp/ipykernel_222554/1651739286.py:17: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.

